# Lesson 12b: Exploration Strategies Practical

**Learning Objectives:**
- Implement multi-armed bandit algorithms (ε-greedy, UCB, Thompson Sampling)
- Build count-based exploration with pseudo-counts
- Implement Intrinsic Curiosity Module (ICM)
- Create Random Network Distillation (RND) for exploration
- Compare exploration methods on sparse reward tasks
- Visualize exploration behavior and coverage

**Prerequisites:** Q-Learning (4b), DQN (7b), Exploration Theory (12a)

**Environment:** Google Colab with GPU support recommended

In [ ]:
# Install dependencies
!pip install gymnasium numpy matplotlib torch tqdm -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import defaultdict, deque
import random
import gymnasium as gym
from tqdm import tqdm
import copy
from typing import Tuple

# Set random seeds
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Multi-Armed Bandits

In [ ]:
class MultiArmedBandit:
    """Simple multi-armed bandit testbed."""
    
    def __init__(self, n_arms: int = 10, mean_reward: float = 0.0, std_reward: float = 1.0):
        """Each arm has Gaussian reward N(μ, 1)."""
        self.n_arms = n_arms
        # Sample true mean for each arm
        self.true_means = np.random.normal(mean_reward, std_reward, n_arms)
        self.best_arm = np.argmax(self.true_means)
        self.best_mean = self.true_means[self.best_arm]
    
    def pull(self, arm: int) -> float:
        """Pull arm, receive reward from N(μ_arm, 1)."""
        return np.random.normal(self.true_means[arm], 1.0)
    
    def get_regret(self, arm: int) -> float:
        """Regret for pulling this arm."""
        return self.best_mean - self.true_means[arm]


# Test bandit
bandit = MultiArmedBandit(n_arms=10)
print(f"True arm means: {bandit.true_means}")
print(f"Best arm: {bandit.best_arm} (mean={bandit.best_mean:.3f})")
print(f"\nTest pulls:")
for _ in range(3):
    arm = np.random.randint(bandit.n_arms)
    reward = bandit.pull(arm)
    print(f"  Arm {arm}: reward={reward:.3f}")

### 1.1 ε-Greedy

In [ ]:
class EpsilonGreedy:
    """ε-greedy bandit algorithm."""
    
    def __init__(self, n_arms: int, epsilon: float = 0.1, alpha: float = 0.1):
        self.n_arms = n_arms
        self.epsilon = epsilon
        self.alpha = alpha
        self.Q = np.zeros(n_arms)  # Estimated values
        self.N = np.zeros(n_arms)  # Counts
    
    def select_arm(self) -> int:
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_arms)  # Explore
        return np.argmax(self.Q)  # Exploit
    
    def update(self, arm: int, reward: float):
        self.N[arm] += 1
        self.Q[arm] += self.alpha * (reward - self.Q[arm])


print("ε-Greedy algorithm created")

### 1.2 UCB (Upper Confidence Bound)

In [ ]:
class UCB:
    """Upper Confidence Bound (UCB1) algorithm."""
    
    def __init__(self, n_arms: int, c: float = 2.0, alpha: float = 0.1):
        self.n_arms = n_arms
        self.c = c  # Exploration coefficient
        self.alpha = alpha
        self.Q = np.zeros(n_arms)
        self.N = np.zeros(n_arms)
        self.t = 0
    
    def select_arm(self) -> int:
        self.t += 1
        
        # Pull each arm once first
        for arm in range(self.n_arms):
            if self.N[arm] == 0:
                return arm
        
        # UCB: Q(a) + c * sqrt(log(t) / N(a))
        ucb_values = self.Q + self.c * np.sqrt(np.log(self.t) / self.N)
        return np.argmax(ucb_values)
    
    def update(self, arm: int, reward: float):
        self.N[arm] += 1
        self.Q[arm] += self.alpha * (reward - self.Q[arm])


print("UCB algorithm created")

### 1.3 Thompson Sampling

In [ ]:
class ThompsonSampling:
    """Thompson Sampling with Gaussian priors."""
    
    def __init__(self, n_arms: int, prior_mean: float = 0.0, prior_std: float = 1.0):
        self.n_arms = n_arms
        # Posterior parameters (Gaussian)
        self.mu = np.full(n_arms, prior_mean)
        self.sigma = np.full(n_arms, prior_std)
        self.N = np.zeros(n_arms)
        self.sum_rewards = np.zeros(n_arms)
    
    def select_arm(self) -> int:
        # Sample from posterior for each arm
        samples = np.random.normal(self.mu, self.sigma)
        return np.argmax(samples)
    
    def update(self, arm: int, reward: float):
        self.N[arm] += 1
        self.sum_rewards[arm] += reward
        
        # Update posterior (Bayesian update for known variance)
        # Assuming reward variance = 1.0
        n = self.N[arm]
        mean_reward = self.sum_rewards[arm] / n
        
        # Posterior: N(μ', σ'²)
        prior_precision = 1.0 / (self.sigma[arm] ** 2)
        likelihood_precision = n
        
        posterior_precision = prior_precision + likelihood_precision
        posterior_variance = 1.0 / posterior_precision
        
        self.mu[arm] = posterior_variance * (likelihood_precision * mean_reward)
        self.sigma[arm] = np.sqrt(posterior_variance)


print("Thompson Sampling algorithm created")

In [ ]:
def run_bandit_experiment(algorithm, bandit, n_steps: int = 1000):
    """Run bandit experiment and track performance."""
    rewards = []
    regrets = []
    
    for _ in range(n_steps):
        arm = algorithm.select_arm()
        reward = bandit.pull(arm)
        algorithm.update(arm, reward)
        
        rewards.append(reward)
        regrets.append(bandit.get_regret(arm))
    
    return np.array(rewards), np.cumsum(regrets)


# Compare algorithms
n_steps = 2000
n_runs = 100

algorithms = {
    'ε-greedy (ε=0.1)': lambda: EpsilonGreedy(10, epsilon=0.1),
    'UCB (c=2)': lambda: UCB(10, c=2.0),
    'Thompson Sampling': lambda: ThompsonSampling(10)
}

results = {name: {'rewards': [], 'regrets': []} for name in algorithms}

print("Running bandit experiments...")
for name, algo_fn in tqdm(algorithms.items()):
    for _ in range(n_runs):
        bandit = MultiArmedBandit(n_arms=10)
        algo = algo_fn()
        rewards, regrets = run_bandit_experiment(algo, bandit, n_steps)
        results[name]['rewards'].append(rewards)
        results[name]['regrets'].append(regrets)

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Average reward
for name in algorithms:
    mean_rewards = np.mean(results[name]['rewards'], axis=0)
    ax1.plot(mean_rewards, label=name, alpha=0.8)
ax1.set_xlabel('Steps')
ax1.set_ylabel('Average Reward')
ax1.set_title('Multi-Armed Bandit Performance')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Cumulative regret
for name in algorithms:
    mean_regrets = np.mean(results[name]['regrets'], axis=0)
    ax2.plot(mean_regrets, label=name, alpha=0.8)
ax2.set_xlabel('Steps')
ax2.set_ylabel('Cumulative Regret')
ax2.set_title('Cumulative Regret Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/bandit_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFinal cumulative regret:")
for name in algorithms:
    final_regret = np.mean([r[-1] for r in results[name]['regrets']])
    print(f"  {name}: {final_regret:.1f}")

## 2. Count-Based Exploration for GridWorld

In [ ]:
class SparseRewardGridWorld(gym.Env):
    """Simple gridworld with sparse reward."""
    
    def __init__(self, size: int = 10, goal_pos: Tuple[int, int] = (9, 9)):
        super().__init__()
        self.size = size
        self.goal_pos = np.array(goal_pos)
        self.start_pos = np.array([0, 0])
        self.pos = self.start_pos.copy()
        
        # Actions: 0=up, 1=right, 2=down, 3=left
        self.action_space = gym.spaces.Discrete(4)
        self.observation_space = gym.spaces.Box(0, size-1, shape=(2,), dtype=np.int32)
    
    def reset(self, seed=None):
        super().reset(seed=seed)
        self.pos = self.start_pos.copy()
        return self.pos.copy(), {}
    
    def step(self, action):
        # Move
        if action == 0:  # up
            self.pos[0] = max(0, self.pos[0] - 1)
        elif action == 1:  # right
            self.pos[1] = min(self.size - 1, self.pos[1] + 1)
        elif action == 2:  # down
            self.pos[0] = min(self.size - 1, self.pos[0] + 1)
        elif action == 3:  # left
            self.pos[1] = max(0, self.pos[1] - 1)
        
        # Reward: +1 at goal, 0 elsewhere (very sparse!)
        done = np.array_equal(self.pos, self.goal_pos)
        reward = 1.0 if done else 0.0
        
        return self.pos.copy(), reward, done, False, {}


class CountBasedQAgent:
    """Q-learning with count-based exploration bonus."""
    
    def __init__(self, n_states: int, n_actions: int,
                 alpha: float = 0.1, gamma: float = 0.99,
                 epsilon: float = 0.1, bonus_coef: float = 0.1):
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.bonus_coef = bonus_coef
        
        self.Q = defaultdict(lambda: np.zeros(n_actions))
        self.N = defaultdict(lambda: np.zeros(n_actions))  # Visit counts
    
    def state_to_key(self, state):
        """Convert state to hashable key."""
        return tuple(state)
    
    def get_exploration_bonus(self, state, action):
        """Compute count-based bonus: β / sqrt(N(s,a) + 1)."""
        key = self.state_to_key(state)
        count = self.N[key][action]
        return self.bonus_coef / np.sqrt(count + 1)
    
    def select_action(self, state):
        key = self.state_to_key(state)
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return np.argmax(self.Q[key])
    
    def update(self, state, action, reward, next_state, done):
        key = self.state_to_key(state)
        next_key = self.state_to_key(next_state)
        
        # Count-based bonus
        bonus = self.get_exploration_bonus(state, action)
        augmented_reward = reward + bonus
        
        # Q-learning update
        td_target = augmented_reward + (0 if done else self.gamma * np.max(self.Q[next_key]))
        self.Q[key][action] += self.alpha * (td_target - self.Q[key][action])
        
        # Update count
        self.N[key][action] += 1
    
    def get_state_visits(self):
        """Get state visit heatmap."""
        visits = defaultdict(int)
        for key in self.N:
            visits[key] = self.N[key].sum()
        return visits


print("Count-based Q-learning agent created")

In [ ]:
def train_gridworld(agent, env, n_episodes: int = 500, max_steps: int = 100):
    """Train agent on gridworld."""
    episode_rewards = []
    episode_lengths = []
    
    for episode in tqdm(range(n_episodes)):
        state, _ = env.reset()
        total_reward = 0
        
        for step in range(max_steps):
            action = agent.select_action(state)
            next_state, reward, done, _, _ = env.step(action)
            
            agent.update(state, action, reward, next_state, done)
            
            total_reward += reward
            state = next_state
            
            if done:
                break
        
        episode_rewards.append(total_reward)
        episode_lengths.append(step + 1)
    
    return episode_rewards, episode_lengths


# Train with and without count-based exploration
env = SparseRewardGridWorld(size=10, goal_pos=(9, 9))

print("Training WITHOUT count-based bonus...")
agent_no_bonus = CountBasedQAgent(
    n_states=100, n_actions=4, bonus_coef=0.0
)
rewards_no_bonus, _ = train_gridworld(agent_no_bonus, env, n_episodes=500)

print("\nTraining WITH count-based bonus...")
agent_with_bonus = CountBasedQAgent(
    n_states=100, n_actions=4, bonus_coef=0.5
)
rewards_with_bonus, _ = train_gridworld(agent_with_bonus, env, n_episodes=500)

# Plot comparison
plt.figure(figsize=(10, 5))
window = 20
plt.plot(np.convolve(rewards_no_bonus, np.ones(window)/window, mode='valid'),
         label='No exploration bonus', alpha=0.8)
plt.plot(np.convolve(rewards_with_bonus, np.ones(window)/window, mode='valid'),
         label='Count-based bonus', alpha=0.8)
plt.xlabel('Episode')
plt.ylabel('Return (smoothed)')
plt.title('Count-Based Exploration on Sparse Reward GridWorld')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('/tmp/count_based_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSuccess rate (last 100 episodes):")
print(f"  Without bonus: {np.mean(rewards_no_bonus[-100:]):.2%}")
print(f"  With bonus: {np.mean(rewards_with_bonus[-100:]):.2%}")

## 3. Intrinsic Curiosity Module (ICM)

In [ ]:
class ICMNetwork(nn.Module):
    """Intrinsic Curiosity Module: Feature encoder + Forward/Inverse models."""
    
    def __init__(self, state_dim: int, action_dim: int, feature_dim: int = 128):
        super(ICMNetwork, self).__init__()
        
        # Feature encoder φ(s)
        self.encoder = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, feature_dim)
        )
        
        # Forward model: (φ(s), a) → φ(s')
        self.forward_model = nn.Sequential(
            nn.Linear(feature_dim + action_dim, 256),
            nn.ReLU(),
            nn.Linear(256, feature_dim)
        )
        
        # Inverse model: (φ(s), φ(s')) → a
        self.inverse_model = nn.Sequential(
            nn.Linear(feature_dim * 2, 256),
            nn.ReLU(),
            nn.Linear(256, action_dim)
        )
    
    def encode(self, state):
        """Encode state to feature space."""
        return self.encoder(state)
    
    def forward(self, state, action, next_state):
        """Compute forward and inverse model outputs."""
        # Encode states
        phi_s = self.encode(state)
        phi_s_next = self.encode(next_state)
        
        # Forward model: predict next feature
        action_onehot = F.one_hot(action, num_classes=self.inverse_model[-1].out_features).float()
        forward_input = torch.cat([phi_s, action_onehot], dim=-1)
        pred_phi_next = self.forward_model(forward_input)
        
        # Inverse model: predict action
        inverse_input = torch.cat([phi_s, phi_s_next], dim=-1)
        pred_action = self.inverse_model(inverse_input)
        
        return pred_phi_next, phi_s_next, pred_action
    
    def get_intrinsic_reward(self, state, action, next_state):
        """Compute intrinsic reward (forward model error)."""
        with torch.no_grad():
            pred_phi_next, phi_s_next, _ = self.forward(state, action, next_state)
            intrinsic_reward = 0.5 * ((pred_phi_next - phi_s_next) ** 2).sum(dim=-1)
        return intrinsic_reward


# Test ICM
icm = ICMNetwork(state_dim=4, action_dim=2, feature_dim=128).to(device)
test_state = torch.randn(10, 4).to(device)
test_action = torch.randint(0, 2, (10,)).to(device)
test_next_state = torch.randn(10, 4).to(device)

pred_phi, phi_next, pred_action = icm(test_state, test_action, test_next_state)
intrinsic_reward = icm.get_intrinsic_reward(test_state, test_action, test_next_state)

print(f"ICM outputs:")
print(f"  Predicted feature: {pred_phi.shape}")
print(f"  True feature: {phi_next.shape}")
print(f"  Predicted action: {pred_action.shape}")
print(f"  Intrinsic reward: {intrinsic_reward.shape}")
print(f"  Mean intrinsic reward: {intrinsic_reward.mean().item():.4f}")

In [ ]:
def train_icm(icm, states, actions, next_states, optimizer, beta: float = 0.2):
    """Train ICM on batch of transitions."""
    pred_phi_next, phi_s_next, pred_action = icm(states, actions, next_states)
    
    # Forward model loss
    forward_loss = F.mse_loss(pred_phi_next, phi_s_next.detach())
    
    # Inverse model loss
    inverse_loss = F.cross_entropy(pred_action, actions)
    
    # Combined loss
    total_loss = (1 - beta) * inverse_loss + beta * forward_loss
    
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    
    return {
        'total_loss': total_loss.item(),
        'forward_loss': forward_loss.item(),
        'inverse_loss': inverse_loss.item()
    }


# Test training
optimizer = optim.Adam(icm.parameters(), lr=1e-3)
losses = train_icm(icm, test_state, test_action, test_next_state, optimizer)
print(f"\nICM training losses:")
for key, value in losses.items():
    print(f"  {key}: {value:.4f}")

## 4. Random Network Distillation (RND)

In [ ]:
class RNDNetwork(nn.Module):
    """Random Network Distillation for exploration."""
    
    def __init__(self, state_dim: int, feature_dim: int = 128):
        super(RNDNetwork, self).__init__()
        
        # Target network (fixed, random initialization)
        self.target = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, feature_dim)
        )
        
        # Predictor network (trained)
        self.predictor = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, feature_dim)
        )
        
        # Freeze target network
        for param in self.target.parameters():
            param.requires_grad = False
    
    def forward(self, state):
        """Compute target and predicted features."""
        with torch.no_grad():
            target_feature = self.target(state)
        pred_feature = self.predictor(state)
        return target_feature, pred_feature
    
    def get_intrinsic_reward(self, state):
        """Compute intrinsic reward (prediction error)."""
        with torch.no_grad():
            target_feature, pred_feature = self.forward(state)
            intrinsic_reward = ((target_feature - pred_feature) ** 2).mean(dim=-1)
        return intrinsic_reward
    
    def train_step(self, states, optimizer):
        """Train predictor to match target."""
        target_feature, pred_feature = self.forward(states)
        loss = F.mse_loss(pred_feature, target_feature.detach())
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        return loss.item()


# Test RND
rnd = RNDNetwork(state_dim=4, feature_dim=128).to(device)
test_states = torch.randn(10, 4).to(device)

target_feat, pred_feat = rnd(test_states)
intrinsic_reward = rnd.get_intrinsic_reward(test_states)

print(f"RND outputs:")
print(f"  Target feature: {target_feat.shape}")
print(f"  Predicted feature: {pred_feat.shape}")
print(f"  Intrinsic reward: {intrinsic_reward.shape}")
print(f"  Mean intrinsic reward: {intrinsic_reward.mean().item():.4f}")

# Test training
rnd_optimizer = optim.Adam(rnd.predictor.parameters(), lr=1e-4)
loss = rnd.train_step(test_states, rnd_optimizer)
print(f"\nRND training loss: {loss:.4f}")

## 5. Comparison on CartPole with Delayed Reward

In [ ]:
class DelayedRewardWrapper(gym.Wrapper):
    """Wrapper that delays reward until episode end (sparse reward)."""
    
    def __init__(self, env):
        super().__init__(env)
        self.total_reward = 0
    
    def reset(self, **kwargs):
        self.total_reward = 0
        return self.env.reset(**kwargs)
    
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.total_reward += reward
        
        # Only give reward at episode end
        sparse_reward = self.total_reward if (terminated or truncated) else 0.0
        
        return obs, sparse_reward, terminated, truncated, info


# Create sparse reward environment
env = gym.make('CartPole-v1')
sparse_env = DelayedRewardWrapper(env)

print(f"Created sparse reward CartPole")
print(f"  State dim: {env.observation_space.shape[0]}")
print(f"  Action dim: {env.action_space.n}")

In [ ]:
class SimpleQLearning:
    """Simple Q-learning with function approximation."""
    
    def __init__(self, state_dim: int, n_actions: int,
                 lr: float = 1e-3, gamma: float = 0.99, epsilon: float = 0.1):
        self.n_actions = n_actions
        self.gamma = gamma
        self.epsilon = epsilon
        
        # Simple Q-network
        self.q_network = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions)
        ).to(device)
        
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        self.buffer = deque(maxlen=10000)
    
    def select_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = self.q_network(state_tensor)
            return q_values.argmax().item()
    
    def update(self, state, action, reward, next_state, done, intrinsic_reward=0.0):
        self.buffer.append((state, action, reward, next_state, done, intrinsic_reward))
        
        if len(self.buffer) < 128:
            return {}
        
        # Sample batch
        batch = random.sample(self.buffer, 128)
        states, actions, rewards, next_states, dones, int_rewards = zip(*batch)
        
        states = torch.FloatTensor(states).to(device)
        actions = torch.LongTensor(actions).to(device)
        rewards = torch.FloatTensor(rewards).to(device)
        next_states = torch.FloatTensor(next_states).to(device)
        dones = torch.FloatTensor(dones).to(device)
        int_rewards = torch.FloatTensor(int_rewards).to(device)
        
        # Augment rewards with intrinsic motivation
        total_rewards = rewards + int_rewards
        
        # Q-learning
        q_values = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze()
        with torch.no_grad():
            next_q_values = self.q_network(next_states).max(1)[0]
            targets = total_rewards + (1 - dones) * self.gamma * next_q_values
        
        loss = F.mse_loss(q_values, targets)
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        return {'q_loss': loss.item()}


print("Simple Q-learning agent created")

In [ ]:
def train_with_exploration(env, agent, exploration_module=None,
                          n_episodes: int = 200, intrinsic_coef: float = 0.01):
    """Train agent with optional exploration module."""
    episode_rewards = []
    
    # Exploration optimizer
    if exploration_module is not None:
        if isinstance(exploration_module, RNDNetwork):
            exp_optimizer = optim.Adam(exploration_module.predictor.parameters(), lr=1e-4)
        else:  # ICM
            exp_optimizer = optim.Adam(exploration_module.parameters(), lr=1e-3)
    
    for episode in tqdm(range(n_episodes)):
        state, _ = env.reset()
        total_reward = 0
        done = False
        
        while not done:
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Compute intrinsic reward
            intrinsic_reward = 0.0
            if exploration_module is not None:
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
                next_state_tensor = torch.FloatTensor(next_state).unsqueeze(0).to(device)
                
                if isinstance(exploration_module, RNDNetwork):
                    intrinsic_reward = exploration_module.get_intrinsic_reward(next_state_tensor).item()
                    exploration_module.train_step(next_state_tensor, exp_optimizer)
                else:  # ICM
                    action_tensor = torch.LongTensor([action]).to(device)
                    intrinsic_reward = exploration_module.get_intrinsic_reward(
                        state_tensor, action_tensor, next_state_tensor
                    ).item()
                    train_icm(exploration_module, state_tensor, action_tensor,
                             next_state_tensor, exp_optimizer)
                
                intrinsic_reward *= intrinsic_coef
            
            # Update agent
            agent.update(state, action, reward, next_state, done, intrinsic_reward)
            
            total_reward += reward
            state = next_state
        
        episode_rewards.append(total_reward)
    
    return episode_rewards


print("Training comparison...")

# Baseline (no exploration bonus)
print("\n1. Baseline (no exploration)")
agent_baseline = SimpleQLearning(4, 2)
rewards_baseline = train_with_exploration(sparse_env, agent_baseline, n_episodes=200)

# RND exploration
print("\n2. With RND exploration")
agent_rnd = SimpleQLearning(4, 2)
rnd_module = RNDNetwork(state_dim=4, feature_dim=64).to(device)
rewards_rnd = train_with_exploration(sparse_env, agent_rnd, rnd_module,
                                    n_episodes=200, intrinsic_coef=0.1)

# ICM exploration
print("\n3. With ICM exploration")
agent_icm = SimpleQLearning(4, 2)
icm_module = ICMNetwork(state_dim=4, action_dim=2, feature_dim=64).to(device)
rewards_icm = train_with_exploration(sparse_env, agent_icm, icm_module,
                                    n_episodes=200, intrinsic_coef=0.05)

In [ ]:
# Plot comparison
plt.figure(figsize=(12, 6))
window = 10

plt.plot(np.convolve(rewards_baseline, np.ones(window)/window, mode='valid'),
         label='Baseline (no exploration)', alpha=0.8, linewidth=2)
plt.plot(np.convolve(rewards_rnd, np.ones(window)/window, mode='valid'),
         label='RND exploration', alpha=0.8, linewidth=2)
plt.plot(np.convolve(rewards_icm, np.ones(window)/window, mode='valid'),
         label='ICM exploration', alpha=0.8, linewidth=2)

plt.xlabel('Episode')
plt.ylabel('Return (smoothed)')
plt.title('Exploration Methods on Sparse Reward CartPole')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('/tmp/exploration_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFinal performance (last 20 episodes):")
print(f"  Baseline: {np.mean(rewards_baseline[-20:]):.1f}")
print(f"  RND: {np.mean(rewards_rnd[-20:]):.1f}")
print(f"  ICM: {np.mean(rewards_icm[-20:]):.1f}")

## Summary

### Implementations Completed

1. **Multi-Armed Bandits**:
   - ε-greedy: Simple uniform exploration
   - UCB: Optimism under uncertainty
   - Thompson Sampling: Bayesian approach
   - Demonstrated optimal regret bounds

2. **Count-Based Exploration**:
   - Simple visit counts for tabular settings
   - Exploration bonuses: $b(s,a) = \beta / \sqrt{N(s,a)}$
   - Significant improvement on sparse reward tasks

3. **Intrinsic Curiosity Module (ICM)**:
   - Feature encoder + forward/inverse models
   - Intrinsic reward from prediction error in feature space
   - Filters out uncontrollable aspects

4. **Random Network Distillation (RND)**:
   - Fixed random target + trained predictor
   - Intrinsic reward from prediction error
   - Simple, effective for very sparse rewards

5. **Comparative Analysis**:
   - All methods improve over baseline
   - RND and ICM particularly effective for sparse rewards
   - Count-based works well in small state spaces

### Key Insights

1. **Directed exploration** (UCB, count-based) beats undirected (ε-greedy)

2. **Intrinsic motivation** crucial for sparse rewards:
   - RND: Simple, robust
   - ICM: More complex, better for some tasks

3. **Hyperparameters matter**:
   - Intrinsic reward coefficient ($\beta$) is critical
   - Too high: distracted by curiosity
   - Too low: insufficient exploration

4. **Scaling**: 
   - Count-based: Good for small spaces
   - ICM/RND: Scale to high dimensions

5. **Combining methods** often works best in practice

### Next Steps

- **Lesson 13**: Multi-agent RL
- Try on Atari with sparse rewards (Montezuma's Revenge)
- Implement episodic memory (NGU-style)
- Add curiosity to existing algorithms (DQN+RND, PPO+ICM)
- Experiment with adaptive intrinsic reward scaling

## Exercises

### Implementation Exercises

1. **Implement decaying ε-greedy** and compare with constant ε

2. **Add Boltzmann exploration** (softmax action selection)

3. **Implement hash-based counts** using locality-sensitive hashing

4. **Build simplified NGU** with episodic memory

5. **Create ensemble of RND networks** for better uncertainty estimation

### Analysis Exercises

6. **Compare UCB with different c values** (0.5, 1, 2, 5)

7. **Analyze ICM feature representations** (visualize with t-SNE)

8. **Study intrinsic reward decay** over training

9. **Measure state coverage** with different exploration methods

10. **Compare RND target network sizes** (32, 64, 128, 256)

### Application Exercises

11. **Apply RND to Atari game** (e.g., Breakout with delayed rewards)

12. **Combine count-based + ICM** bonuses

13. **Implement adaptive β scheduling** (decay intrinsic coefficient over time)

14. **Add RND to PPO** for continuous control

15. **Build curiosity-driven exploration** for robotics task